<a href="https://colab.research.google.com/github/Ramprasath-Meganathan/LoRA-Plus/blob/main/ML_Project_LoRA%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Master toggles - Change to True ONLY when necessary
RUN_INSTALLATION = False
RUN_HEAVY_TRAINING = False
DOWNLOAD_DATASETS = False

In [3]:
#Save Libraries to Google Drive for ease of Installation
from google.colab import drive
import os
import sys

# Create a path for your persistent library storage
lib_path = '/content/drive/MyDrive/colab_env/'

if os.path.exists(lib_path):
  print("Google drive is already mounted and verified. Skipping Authentication...")
else:
  print("Target path not detected. Initializing active Google Drive mount pipeline...")
  drive.mount('/content/drive')

  #Enforce directory creation if its a completely fresh drive
  !mkdir -p {lib_path}
  print("Drive mounted and folder verified.")

# CLEANUP: Remove the slow, unzipped Drive directory from sys.path to avoid circular imports
if lib_path in sys.path:
  sys.path.remove(lib_path)
  print("Removed slow Google Drive path from sys.path to prevent import conflicts.")

Target path not detected. Initializing active Google Drive mount pipeline...
Mounted at /content/drive
Drive mounted and folder verified.


In [4]:
import os
import shutil

# Define paths
local_stage_path = "/content/local_hf_env"
drive_zip_path = "/content/drive/MyDrive/colab_env.zip"

if RUN_INSTALLATION:
  print("Stage 1: Installing ONLY missing essential libraries (peft, evaluate) to ultra-fast local disk...")
  # Delete any old local staging folder if it exists
  if os.path.exists(local_stage_path):
    shutil.rmtree(local_stage_path)

  # Install only the target-specific libraries not included in default Google Colab
  !pip install -U -q peft evaluate --target={local_stage_path}

  print("Stage 2: Compressing environment into a single archive...")
  # Zip the entire folder so it counts as ONE file transfer to Drive
  shutil.make_archive("/content/drive/MyDrive/colab_env", 'zip', local_stage_path)

  print(f"Environment successfully packed and saved to Drive: {drive_zip_path}")

In [ ]:
# High-Speed Local Cache Loader
# Run this on new sessions to quickly load libraries from Drive zip to local container disk in seconds
import os
import shutil
import sys

local_stage_path = "/content/local_hf_env"
drive_zip_path = "/content/drive/MyDrive/colab_env.zip"

# 1. If we _need to build the cache because it doesn't exist yet
if RUN_INSTALLATION:
    print("RUNINSTALLATION is True. Re-building cache... Please run the installation cells above.")
else:
    # 2. If cache exists, load it directly to local disk in under 10 seconds
    if os.path.exists(drive_zip_path):
        print("Zip archive detected on Drive. Extracting to local Colab high-speed disk...")
        if os.path.exists(local_stage_path):
            shutil.rmtree(local_stage_path)
        os.makedirs(local_stage_path, exist_ok=True)

        # Unpack directly to local SSD
        shutil.unpack_archive(drive_zip_path, local_stage_path, 'zip')
        print("Extraction complete.")
    else:
        print("No zip archive found on Drive. Please set RUN_INSTALLATION=True, run the setup cells, and pack the environment first.")

# 3. Always register the local ultra-fast path instead of the slow Google Drive path
if local_stage_path not in sys.path:
    sys.path.insert(0, local_stage_path)
    print(f"Local high-speed path registered: {local_stage_path}")

Zipa rchive detected on Drive. Extracting to local Colab high-speed disk...


In [ ]:
import sys
# Inject our high-speed local package path at the top of python path registry
local_stage_path = "/content/local_hf_env"
if local_stage_path not in sys.path:
    sys.path.insert(0, local_stage_path)

# Necessary Library Imports
import os
import math
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import peft
from datasets import load_dataset
from torch.optim import AdamW
from google.colab import drive
from peft import get_peft_model, LoraConfig, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
print("PyTorch", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

In [ ]:
# Verify GPU environment (Google Colab T4 16GB limit)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive for saving checkpoints to protect against session timeouts
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_Plus_Checkpoints/"
if os.path.exists('/content/drive/MyDrive'):
  print("Google drive is already verified. Skipping mount request")
else:
  print("Target mount point not detected. Mounting Google Drive...")
  drive.mount('/content/drive')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the dataset
dataset = load_dataset("nyu-mll/glue", "sst2")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
  return tokenizer(
      batch["sentence"],
      truncation=True,
      max_length=128
  )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["sentence", "idx"]
)

tokenized_dataset = tokenized_dataset.rename_column(
    "label", "labels"
)

tokenized_dataset.set_format("torch")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokens = tokenizer.tokenize("The movie is amazing.")
print(tokens)

encoded = tokenizer("The movie is amazing.")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

In [ ]:
print(tokenized_dataset)

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["query","value"],
    modules_to_save=["classifier"],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


In [ ]:
import torch

trainable_parameters = []

for parameter in model.parameters():
    if parameter.requires_grad:
        trainable_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-4,
    weight_decay=0.01
)

In [ ]:
lora_a_parameters = []
lora_b_parameters = []
other_parameters = []

for name, parameter in model.named_parameters():
  if not parameter.requires_grad:
    continue

  if "lora_A" in name:
    lora_a_parameters.append(parameter)
  elif "lora_B" in name:
    lora_b_parameters.append(parameter)
  else:
    other_parameters.append(parameter)

base_lr = 1e-4
lambda_ratio = 8

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_a_parameters,
            "lr": base_lr
        },
        {
            "params": lora_b_parameters,
            "lr": base_lr * lambda_ratio
        },
        {
            "params": other_parameters,
        }
    ],
    weight_decay=0.01
)

In [ ]:
# Force kill the VM runtime container right after execution finishes
from google.colab import runtime
runtime.unassign()